In [1]:
# %load_ext autoreload
# %autoreload 2

# from utgen.raggraph.utils import get_node_context
# from utgen.test_generation_crew.crew import TestGenerationCrew

In [29]:
imports1 = "from utgen.raggraph.walker import iter_python_files"
test1 = '''
# Test cases based on Incoming Edges analysis
def test_iter_python_files_basic_usage(tmp_path):
    """
    Test basic usage pattern found in build_graph_from_directory call.
    Verifies that the function yields .py files correctly. FAIL
    """
    # Create test directory structure
    (tmp_path / "module1").mkdir()
    (tmp_path / "module1" / "__init__.py").touch()
    (tmp_path / "module1" / "file1.py").touch()
    (tmp_path / "module2").mkdir()
    (tmp_path / "module2" / "file2.py").touch()

    # Test default behavior (skip_init=True)
    result = list(iter_python_files(tmp_path))
    assert len(result) == 2
    assert "module1/file1.py" in result
    assert "module2/file2.py" in result

    # Test skip_init=False
    result = list(iter_python_files(tmp_path, skip_init=False))
    assert len(result) == 3
    assert "module1/__init__.py" in result
'''
imports2 = "from utgen.raggraph.walker import iter_python_files"
test2 = '''
def test_iter_python_files_empty_directory(tmp_path):
    """
    Test edge case: empty directory should yield no files. PASS
    """
    result = list(iter_python_files(tmp_path))
    assert len(result) == 0
'''
imports3 = "from utgen.raggraph.walker import iter_python_files"
test3 = '''
def test_iter_python_files_no_python_files(tmp_path):
    """
    Test edge case: directory with no .py files. PASS
    """
    (tmp_path / "module1").mkdir()
    (tmp_path / "module1" / "file1.txt").touch()
    (tmp_path / "module1" / "file2.md").touch()

    result = list(iter_python_files(tmp_path))
    assert len(result) == 0
'''
imports4 = "from utgen.raggraph.walker import iter_python_files"
test4 = '''
def test_iter_python_files_nested_directories(tmp_path):
    """
    Test nested directory structure handling. FAIL
    """
    (tmp_path / "level1").mkdir()
    (tmp_path / "level1" / "level2").mkdir()
    (tmp_path / "level1" / "level2" / "file.py").touch()
    (tmp_path / "level1" / "file.py").touch()

    result = list(iter_python_files(tmp_path))
    assert len(result) == 2
    assert "level1/file.py" in result
    assert "level1/level2/file.py" in result
'''
imports5 = "from utgen.raggraph.walker import iter_python_files"
test5 = '''
def test_iter_python_files_with_non_standard_names(tmp_path):
    """
    Test handling of non-standard .py file names. FAIL
    """
    (tmp_path / "module1").mkdir()
    (tmp_path / "module1" / "test_file.py").touch()
    (tmp_path / "module1" / "test-file.py").touch()
    (tmp_path / "module1" / "test_file.PY").touch()

    result = list(iter_python_files(tmp_path))
    assert len(result) == 2
    assert "module1/test_file.py" in result
    assert "module1/test-file.py" in result
    assert "module1/test_file.PY" not in result  # Case-sensitive check
'''
imports6 = "from utgen.raggraph.walker import iter_python_files"
test6 = '''
def test_iter_python_files_with_symlinks(tmp_path):
    """
    Test handling of symbolic links. FAIL
    """
    (tmp_path / "module1").mkdir()
    (tmp_path / "module1" / "file.py").touch()
    (tmp_path / "module2").mkdir()
    (tmp_path / "module2" / "link_to_file.py").symlink_to(tmp_path / "module1" / "file.py")

    result = list(iter_python_files(tmp_path))
    assert len(result) == 2
    assert "module1/file.py" in result
    assert "module2/link_to_file.py" in result
'''
imports7 = "from utgen.raggraph.walker import iter_python_files \nimport pytest \nfrom unittest.mock import patch \nimport os"
test7 = '''
@patch('utgen.raggraph.walker.os.walk')
def test_iter_python_files_os_walk_error(mock_walk):
    """
    Test error handling when os.walk raises an exception. PASS
    """
    mock_walk.side_effect = OSError("Permission denied")

    with pytest.raises(OSError):
        list(iter_python_files("/some/path"))
'''
imports8 = "from utgen.raggraph.walker import iter_python_files"
test8 = '''
def test_iter_python_files_with_hidden_files(tmp_path):
    """
    Test handling of hidden files (starting with .). FAIL
    """
    (tmp_path / "module1").mkdir()
    (tmp_path / "module1" / ".hidden.py").touch()
    (tmp_path / "module1" / "visible.py").touch()

    result = list(iter_python_files(tmp_path))
    assert len(result) == 2
    assert "module1/.hidden.py" in result
    assert "module1/visible.py" in result
'''

In [30]:
# Simulació de l'output del LLM (llista de tuples)
llista_output_llm = [
    (imports1, test1),
    (imports2, test2),
    (imports3, test3),
    (imports4, test4),
    (imports5, test5),
    (imports6, test6),
    (imports7, test7),
    (imports8, test8),
]

In [47]:
import subprocess
import os

def validar_test_individual(import_code, test_code):
    """
    Prova un test en un fitxer temporal. 
    Retorna True si el test passa, False si no.
    """
    temp_filename = "temp_validation.py"
    full_code = f"{import_code}\n\n{test_code}"

    with open(temp_filename, "w") as f:
        f.write(full_code)
    
    # Executem pytest. -q per silenci, --tb=no per no embrutar la consola
    result = subprocess.run(["pytest", temp_filename, "-q", "--tb=no"], capture_output=True)
    
    if os.path.exists(temp_filename):
        os.remove(temp_filename)
        
    return result.returncode == 0

def guardar_i_netejar_tests(tests_valids, desti="../tests/test_generats_llm.py"):
    """
    tests_valids: Llista de tuples [(import, codi), ...]
    """
    if not tests_valids:
        print("No hi ha tests vàlids per guardar.")
        return

    os.makedirs(os.path.dirname(desti), exist_ok=True)

    # 1. SEPAREM EN DOS BLOCS
    bloc_imports = []
    bloc_tests = []
    
    for imp, code in tests_valids:
        bloc_imports.append(imp.strip())
        bloc_tests.append(code.strip())
    
    # 2. CONCATENEM: primer TOTS els imports, després els tests
    contingut_final = "\n".join(bloc_imports) + "\n\n" + "\n\n".join(bloc_tests)

    with open(desti, "w") as f:
        f.write(contingut_final)

    # 2. Passem Ruff per endreçar-ho tot
    print(f"Netejant {desti} amb Ruff...")
    
    # Ordenar imports i eliminar duplicats (isort)
    subprocess.run(["ruff", "check", "--select", "I", "--fix", desti], capture_output=True)
    # # Eliminar variables/imports no usats i altres errors comuns
    subprocess.run(["ruff", "check", "--fix", desti], capture_output=True)
    # # Formatar codi (estil Black)
    subprocess.run(["ruff", "format", desti], capture_output=True)
    
    print(f"✅ Procés finalitzat. Fitxer guardat a: {desti}")

In [48]:
tests_que_han_passat = []

for imp, code in llista_output_llm:
    if validar_test_individual(imp, code):
        print(f"✅ Test acceptat")
        tests_que_han_passat.append((imp, code))
    else:
        print(f"❌ Test rebutjat")

# Guardem i deixem el fitxer perfecte
guardar_i_netejar_tests(tests_que_han_passat)

❌ Test rebutjat
✅ Test acceptat
✅ Test acceptat
❌ Test rebutjat
❌ Test rebutjat
❌ Test rebutjat
✅ Test acceptat
❌ Test rebutjat
Netejant ../tests/test_generats_llm.py amb Ruff...
✅ Procés finalitzat. Fitxer guardat a: ../tests/test_generats_llm.py
